In [ ]:
# In colab run this cell first to setup the file structure!
%cd /content
!rm -rf MOL518-Intro-to-Data-Analysis

!git clone https://github.com/shaevitz/MOL518-Intro-to-Data-Analysis.git
%cd MOL518-Intro-to-Data-Analysis/Lecture_34

# MOL518 Lecture 34: Convolutions and Feature Extraction

## Lecture Outline
- Convolutions in 1D and 2D
- Padding and edge handling
- Derivative filters and Sobel operators
- Second derivatives and the Hessian
- Laplacian of Gaussian and blob detection
- Template matching as feature detection

Convolution is one of the main workhorses of image analysis. You can think of it as applying the same small recipe to neighboring values in an image over and over again. 

In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from scipy import ndimage, signal

plt.rcParams['figure.dpi'] = 120
plt.rcParams['image.cmap'] = 'gray'

media_dir = Path('media')

# We wil use this later in the lecture...
def add_plot_box(ax, linewidth=1.1, color='0.2'):
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(linewidth)
        spine.set_edgecolor(color)

## Convolutions in 1D

A convolution works by sliding a small window along a signal. At each position, it multiplies each value in the window by a corresponding weight, sums the results to produce a single output value, then advances one step and repeats.

Those weights are called a **kernel**. For example, the kernel `[1/3, 1/3, 1/3]` computes a simple three-point moving average: each output value is the mean of the current sample and its two immediate neighbors.

Other kernel shapes smooth in different ways. The 5-point kernel `[1/16, 1/4, 6/16, 1/4, 1/16]` performs a Gaussian-weighted moving average. The central data point carries the most weight, the immediate neighbors somewhat less, and the outermost samples the least. Because distant samples are downweighted rather than ignored, this produces a smoother result than a flat average kernel of the same width.

### Derivatives as Convolutions

We can also use a convolution to measure how the data changes, i.e. we can take a derivative. If you remember your calculus class, the definition of the derivative was

$$
\frac{df}{dx} = \lim_{\Delta x \to 0} \frac{f(x + \Delta x) - f(x)}{\Delta x}.
$$

For discretely sampled data, we cannot take $\Delta x \to 0$, but we can approximate the derivative by comparing neighboring samples. The **central difference** formula is

$$
\frac{df}{dx} \approx \frac{f(x+\Delta x) - f(x-\Delta x)}{2\Delta x}.
$$

If we first compute

$$
\frac{f(x+\Delta x) - f(x-\Delta x)}{2},
$$

that corresponds to convolving with the kernel `[-1/2, 0, 1/2]` (with the convolution flip convention noted below). Dividing that result by $\Delta x$ gives the actual derivative $df/dx$.

*Note:* convolution has a notation quirk: by definition it flips the kernel before sliding. For symmetric kernels this does not matter, but for odd-symmetric derivative kernels it changes the sign. That is why the code stores the derivative kernel in reversed order when using `convolve`.

In [ ]:
# The derivative of sin(x) is cos(x), let's check with a convolution
x = np.linspace(0, 2 * np.pi, 200)
signal_sine = np.sin(x)
true_deriv = np.cos(x)

# Central-difference kernel stored in reverse order because `convolve`` flips it.
deriv_kernel = np.array([0.5, 0.0, -0.5])

# Convert finite-difference output to an actual derivative by dividing by dx.
dx = x[1] - x[0]
deriv_response = signal.convolve(signal_sine, deriv_kernel, mode='same') / dx


fig, axes = plt.subplots(2, 1, figsize=(5, 4), sharex=True)
axes[0].plot(x, signal_sine)
axes[0].set_ylabel('f(x) = sin(x)')
axes[1].plot(x, deriv_response, label='Convolution derivative', linewidth=2)
axes[1].plot(x, true_deriv, '--', color='C3', label='True derivative (cos)', linewidth=1.5)
axes[1].set_ylabel("df/dx")
axes[1].set_xlabel('x')
axes[1].legend()

plt.tight_layout()

### Padding and Edge Handling in Convolutions

Notice that in the derivative example above the first and last points look odd. That is a boundary artifact caused by how convolutions handle missing neighbors at the edges.

The sliding-window idea works cleanly in the middle of a signal, but at the boundaries the kernel hangs off the data. At the very first point, for example, a 3-point kernel needs one value to the left that does not exist.

**Padding** is the rule used to invent those missing values. For a concrete example, take a short sequence `[2, 4, 6, 8, 10]` and pad by one value on each side:

| Padding rule | Result outside range | Example padded sequence | Boundary behavior |
|---|---|---|---|
| Zero padding | Values outside are 0 | `[0, 2, 4, 6, 8, 10, 0]` | Can introduce artificial edge spikes/dips |
| Edge padding | Repeat endpoint values | `[2, 2, 4, 6, 8, 10, 10]` | Gentler endpoint behavior |
| Reflect padding | Mirror nearby values | `[4, 2, 4, 6, 8, 10, 8]` | Smoothest transition at boundaries |

In the code below, we first define the **canonical interior derivative sequence** (where both neighbors exist, so no padding is needed), then compare how each padding rule extends the estimate to the endpoints.

In [ ]:
pad_width = len(deriv_kernel) // 2

# Define the three padded versions explicitly.
padded_zero = np.pad(signal_sine, pad_width, mode='constant', constant_values=0)
padded_edge = np.pad(signal_sine, pad_width, mode='edge')
padded_reflect = np.pad(signal_sine, pad_width, mode='reflect')

# Compute derivative estimates for each padding rule.
deriv_zero = signal.convolve(padded_zero, deriv_kernel, mode='valid') / dx
deriv_edge = signal.convolve(padded_edge, deriv_kernel, mode='valid') / dx
deriv_reflect = signal.convolve(padded_reflect, deriv_kernel, mode='valid') / dx

fig, axes = plt.subplots(2, 1, figsize=(6, 5), sharex=True)

axes[0].plot(x, signal_sine, color='k', linewidth=2)
axes[0].set_ylabel('f(x) = sin(x)')

axes[1].plot(x, true_deriv, 'k--', linewidth=2, label='True derivative: cos(x)')
axes[1].plot(x, deriv_zero, linewidth=1.8, label='Zero padding')
axes[1].plot(x, deriv_edge, linewidth=1.8, label='Repeat edge')
axes[1].plot(x, deriv_reflect, linewidth=1.8, label='Reflect')

# Highlight first/last points where padding assumptions differ.
for y, c in [(deriv_zero, 'C0'), (deriv_edge, 'C1'), (deriv_reflect, 'C2')]:
    axes[1].scatter([x[0], x[-1]], [y[0], y[-1]], color=c, s=30, zorder=3)

axes[1].set_ylabel('df/dx estimate')
axes[1].set_xlabel('x')
axes[1].legend(ncol=2, fontsize=9)

plt.tight_layout()

## Convolutions in 2D

For an image $I(x, y)$, a 2D convolution applies the same local weighted sum over a *2D neighborhood*. 
- Recipe:
    - Place the kernel over a pixel and its neighbors.

    - Multiply each neighboring pixel by the corresponding kernel value.
    - Sum the results.
    - Move the kernel across the image (horiz and vert), repeating steps 1-3.

Different kernels emphasize different image properties:

- **Identity kernel**: returns the original image.
  $$
  \begin{bmatrix}
  0 & 0 & 0 \\
  0 & 1 & 0 \\
  0 & 0 & 0
  \end{bmatrix}
  $$

- **Shift-left kernel**: moves the image one pixel to the left.
  $$
  \begin{bmatrix}
  0 & 0 & 0 \\
  0 & 0 & 1 \\
  0 & 0 & 0
  \end{bmatrix}
  $$

- **Mean filter**: blurs the image by averaging a 3 x 3 neighborhood.
  $$
  \frac{1}{9}
  \begin{bmatrix}
  1 & 1 & 1 \\
  1 & 1 & 1 \\
  1 & 1 & 1
  \end{bmatrix}
  $$

- **Sharpening kernel**: subtracts a local average and emphasizes the center pixel.
  $$
  \begin{bmatrix}
  0 & -1 & 0 \\
  -1 & 5 & -1 \\
  0 & -1 & 0
  \end{bmatrix}
  $$

We can demonstrate all of these on a simple synthetic image.

In [ ]:
# Apply several 2D kernels to the same simple image.
image = np.ones((90, 120), dtype=float)
image[20:70, 30:90] = 0.35
image[35:55, 50:70] = 0.05
image[12:18, 100:106] = 0.0

identity = np.array(
    [
        [0, 0, 0],
        [0, 1, 0],
        [0, 0, 0],
    ]
)
shift_left = np.array(
    [
        [0, 0, 0],
        [0, 0, 1],
        [0, 0, 0],
    ]
)
mean_kernel = np.array(
    [
        [1, 1, 1],
        [1, 1, 1],
        [1, 1, 1],
    ],
    dtype=float,
) / 9

sharpen_kernel = np.array(
    [
        [0, -1, 0],
        [-1, 5, -1],
        [0, -1, 0],
    ]
)

outputs = [
    ('Original', image),
    ('Identity', signal.convolve2d(image, identity, mode='same', boundary='symm')),
    ('Shift left (1 px)', signal.convolve2d(image, shift_left, mode='same', boundary='symm')),
    ('Mean blur', signal.convolve2d(image, mean_kernel, mode='same', boundary='symm')),
    ('Sharpen', signal.convolve2d(image, sharpen_kernel, mode='same', boundary='symm')),
]

fig, axes = plt.subplots(1, 5, figsize=(13, 3))
for ax, (title, result) in zip(axes, outputs):
    ax.imshow(result, vmin=0, vmax=1)
    ax.set_title(title)
    ax.set_xticks([])
    ax.set_yticks([])
    add_plot_box(ax)
plt.tight_layout()


## Convolutions in Image Processing

- A mathematical operation used to apply a filter (called a kernel) to an image. 

- Works by sliding a small matrix kernel over the image and computing a weighted sum of pixel values at each location.

    - Place the kernel over a pixel and its neighbors.

    - Multiply each neighboring pixel by the corresponding kernel value.
    - Sum the results.
    - Move the kernel across the image, repeating steps 1-3.

- We will use convolutions for many things, including smoothing, sharpening, edge detection, and feature extraction in images

### 3x3 Mean Filter as a convolution

The **kernel** is defined as:
$$
M = \frac{1}{9}
\begin{bmatrix}
1 & 1 & 1 \\
1 & 1 & 1 \\
1 & 1 & 1
\end{bmatrix}
$$

The filtered image is then given by the convolution of the original image with the kernel:

$$
I'(x,y) = \sum_{i=-1}^{1} \sum_{j=-1}^{1} M(i,j) I(x+i, y+j).
$$

Example Python code:

```python
# Define a 3x3 mean-filter kernel
kernel = np.ones((3, 3), dtype=float) / 9

# Convolve image with the kernel to get a mean-filtered image
# (mode='nearest' handles borders by repeating edge values)
mean_filtered = ndimage.convolve(noisy_image, kernel, mode='nearest')
```


### 3x3 Gaussian Filter as a convolution

To perform a Gaussian filter, we just need to redefine the kernel:
$$
G = \frac{1}{16}
\begin{bmatrix}
1 & 2 & 1 \\
2 & 4 & 2 \\
1 & 2 & 1
\end{bmatrix}
$$

Here we use $\sigma=1$.

In Python, you could calculate from the actual Gaussian function or just put in by hand:

```python
# Define a 3x3 Gaussian kernel
gaussian_kernel = np.array([
    [1, 2, 1],
    [2, 4, 2],
    [1, 2, 1]
], dtype=np.float32) / 16
```


## Exercise 1
Make a filter than shifts the image *downward* over by 2 pixels. This is a little subtle, so think before you code!

In [ ]:
# Your code goes here




## Sobel Filters and Edge Detection

The Sobel operators combine a derivative with a small amount of smoothing. The standard kernels are

$$
G_x = \begin{bmatrix}-1 & 0 & 1 \\ -2 & 0 & 2 \\ -1 & 0 & 1\end{bmatrix}, \qquad
G_y = \begin{bmatrix}-1 & -2 & -1 \\ 0 & 0 & 0 \\ 1 & 2 & 1\end{bmatrix}.
$$

Applying these gives horizontal and vertical gradient estimates. From them we can compute the gradient magnitude and direction:

$$
|\nabla I| = \sqrt{G_x^2 + G_y^2}, \qquad \theta = \operatorname{atan2}(G_y, G_x).
$$

*Note: The gradient $\nabla I$ gives the direction of steepest intensity increase, and its magnitude $|\nabla I|$ gives the edge strength. The function `atan2` returns the gradient angle $\theta$ in a quadrant-aware way.*

In [ ]:
yy, xx = np.mgrid[:140, :140]
edge_image = np.ones((140, 140), dtype=float)
edge_image[(xx - 45) ** 2 + (yy - 45) ** 2 < 18**2] = 0.2
edge_image[(np.abs(xx - 95) < 6) & (yy > 25) & (yy < 115)] = 0.45
edge_image[(yy > 75) & (yy < 110) & (xx > 25) & (xx < 120)] = np.minimum(edge_image[(yy > 75) & (yy < 110) & (xx > 25) & (xx < 120)], 0.65)

sobel_x = ndimage.sobel(edge_image, axis=1, mode='reflect')
sobel_y = ndimage.sobel(edge_image, axis=0, mode='reflect')
gradient_magnitude = np.hypot(sobel_x, sobel_y)
gradient_direction = np.degrees(np.arctan2(sobel_y, sobel_x))

fig, axes = plt.subplots(2, 3, figsize=(11, 7))
for ax, title, result in zip(
    axes.flat,
    ['Input image', 'Sobel x', 'Sobel y', 'Gradient magnitude', 'Gradient direction', 'Direction on strong edges'],
    [
        edge_image,
        sobel_x,
        sobel_y,
        gradient_magnitude,
        gradient_direction,
        np.where(gradient_magnitude > 0.4, gradient_direction, np.nan),
    ],
):
    im = ax.imshow(result)
    ax.set_title(title)
    ax.set_xticks([])
    ax.set_yticks([])
    add_plot_box(ax)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.tight_layout()

## Second Derivatives and the Hessian

The Hessian collects second derivatives:

$$
\mathbf{H} =
\begin{bmatrix}
I_{xx} & I_{xy} \\
I_{yx} & I_{yy}
\end{bmatrix}.
$$

Second derivatives tell us about **curvature**, not just slope:

- If both Hessian eigenvalues are large and have similar magnitude, the neighborhood is blob-like.
- If one eigenvalue is large and the other is small, the neighborhood is more ridge-like or edge-like.

The Hessian is useful for detecting blobs, ridges, and other structured features.

In [ ]:
shape_image = np.zeros((120, 120), dtype=float)
shape_image += np.exp(-((xx[:120, :120] - 35) ** 2 + (yy[:120, :120] - 38) ** 2) / (2 * 9**2))
shape_image += 0.8 * np.exp(-((xx[:120, :120] - 80) ** 2) / (2 * 4**2)) * np.exp(-((yy[:120, :120] - 80) ** 2) / (2 * 20**2))

Ixx = ndimage.gaussian_filter(shape_image, sigma=2, order=(0, 2))
Iyy = ndimage.gaussian_filter(shape_image, sigma=2, order=(2, 0))
Ixy = ndimage.gaussian_filter(shape_image, sigma=2, order=(1, 1))
trace = Ixx + Iyy
determinant = Ixx * Iyy - Ixy**2

fig, axes = plt.subplots(2, 3, figsize=(10, 6.5))
for ax, title, result in zip(
    axes.flat,
    ['Input', r'$I_{xx}$', r'$I_{yy}$', r'$I_{xy}$', 'Trace', 'Determinant'],
    [shape_image, Ixx, Iyy, Ixy, trace, determinant],
):
    im = ax.imshow(result)
    ax.set_title(title)
    ax.set_xticks([])
    ax.set_yticks([])
    add_plot_box(ax)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.tight_layout()

## Laplacian of Gaussian (Mexican Hat) Filter

The **Laplacian** adds second derivatives together, and the **Gaussian** smooths first to suppress noise. Their combination is the Laplacian of Gaussian (LoG):

$$
\nabla^2 G_\sigma * I.
$$

The LoG is often called a **Mexican hat filter** because its shape has a positive center, a negative ring, and then a decay back toward zero. This makes it sensitive to blob-like objects at a characteristic size controlled by $\sigma$.



In [ ]:
sigma = 3.5
half_size = 4 * sigma
n_points = 201

xk = np.linspace(-half_size, half_size, n_points)
yk = np.linspace(-half_size, half_size, n_points)
xx_k, yy_k = np.meshgrid(xk, yk)
r2 = xx_k**2 + yy_k**2

# LoG kernel shape. I've adde a minus sign for visualization (similar to the sign flip in the 1D derivative)
log_kernel = -((r2 - 2 * sigma**2) / sigma**4) * np.exp(-r2 / (2 * sigma**2))

fig = plt.figure(figsize=(7.2, 5.2))
ax = fig.add_subplot(111, projection='3d')
surf = ax.plot_surface(
    xx_k,
    yy_k,
    log_kernel,
    cmap='coolwarm',
    edgecolor='none',
    antialiased=True,
)

ax.set_title(f'3D shape of LoG filter (sigma={sigma:g})')
ax.set_xlabel('x offset')
ax.set_ylabel('y offset')
ax.set_zlabel('Kernel value')
fig.colorbar(surf, ax=ax, shrink=0.65, pad=0.12);

In [ ]:
rng = np.random.default_rng(3)
spot_image = np.zeros((160, 160), dtype=float)
yy_spot, xx_spot = np.mgrid[:160, :160]
centers = [(35, 40), (70, 100), (110, 55), (120, 120)]
for cy, cx in centers:
    spot_image += np.exp(-((xx_spot - cx) ** 2 + (yy_spot - cy) ** 2) / (2 * 5.5**2))
spot_image += 0.08 * rng.normal(size=spot_image.shape)
spot_image = np.clip(spot_image, 0, None)

log_response = ndimage.gaussian_laplace(spot_image, sigma=4)

fig, axes = plt.subplots(1, 2, figsize=(8.5, 3.8))
axes[0].imshow(spot_image)
axes[0].set_title('Synthetic spot image')
axes[0].set_xticks([])
axes[0].set_yticks([])
add_plot_box(axes[0])

im = axes[1].imshow(log_response, cmap='coolwarm')
axes[1].set_title('LoG response')
axes[1].set_xticks([])
axes[1].set_yticks([])
add_plot_box(axes[1])
plt.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)
plt.tight_layout()

## Feature Detection by Template Matching

A convolution or correlation can also do **pattern matching**. If we slide a small template across a larger image and score how well it matches, peaks in the response tell us where that pattern appears.

The figure below illustrates the idea with butterfly detection:

![Template matching example](media/template_matching_example.png)

In practice, this is the same logic behind many matched-filter and template-matching methods.

In [ ]:
# Build a small synthetic template and find it in a larger scene.
def make_bowtie(size=17):
    y, x = np.mgrid[:size, :size]
    x = x - size // 2
    y = y - size // 2
    template = ((np.abs(y) <= -0.7 * np.abs(x) + 5) & (np.abs(x) <= 6)).astype(float)
    template += ((np.abs(x) <= 1) & (np.abs(y) <= 6)).astype(float)
    return np.clip(template, 0, 1)

template = make_bowtie()
scene = np.zeros((90, 120), dtype=float)
placements = [(22, 30), (55, 78)]
for r, c in placements:
    scene[r : r + template.shape[0], c : c + template.shape[1]] += template
scene += 0.2 * rng.random(scene.shape)

correlation = signal.correlate2d(scene, template, mode='same')
peak_mask = correlation == ndimage.maximum_filter(correlation, size=15)
peak_mask &= correlation > 0.75 * correlation.max()
peaks = np.argwhere(peak_mask)

fig, axes = plt.subplots(1, 3, figsize=(12, 3.6))
axes[0].imshow(template)
axes[0].set_title('Template')
axes[0].set_xticks([])
axes[0].set_yticks([])
add_plot_box(axes[0])

axes[1].imshow(scene)
axes[1].set_title('Scene')
axes[1].set_xticks([])
axes[1].set_yticks([])
add_plot_box(axes[1])

im = axes[2].imshow(correlation, cmap='magma')
axes[2].set_title('Correlation map')
axes[2].set_xticks([])
axes[2].set_yticks([])
add_plot_box(axes[2])
plt.colorbar(im, ax=axes[2], fraction=0.046, pad=0.04)

for r, c in peaks:
    axes[1].add_patch(plt.Rectangle((c - template.shape[1] // 2, r - template.shape[0] // 2), template.shape[1], template.shape[0], edgecolor='C3', facecolor='none', linewidth=2))

plt.tight_layout()
